# VRNN Training Notebook
This notebook contains the complete implementation of a Variational Recurrent Neural Network (VRNN) for stock market analysis.

In [ ]:
# Install dependencies if needed (Colab/Kaggle standard environments usually have these, except maybe tqdm update)
!pip install tqdm pandas torch numpy matplotlib

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

## 1. Data Utilities
Implementation of the `FinanceDataset` class for preprocessing financial time series.

In [ ]:
class FinanceDataset(Dataset):
    def __init__(self, T: int, file_path: str, feature_columns: list[str], normalize: bool = True):
        self.T = T
        self.file_path = file_path
        self.feature_columns = feature_columns
        self.normalize = normalize

        df = pd.read_csv(file_path)
        df = df[self.feature_columns]

        # Clean string columns: remove '%' and ',' then convert to float
        for col in self.feature_columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].astype(str).str.replace('%', '', regex=False)
                df[col] = df[col].str.replace(',', '', regex=False)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Remove any rows with NaN values
        df = df.dropna()
        
        self.data = df.values.astype('float32')

        if self.normalize:
            self.data = (self.data - self.data.mean(axis=0)) / (self.data.std(axis=0) + 1e-8)

    def __len__(self):
        return len(self.data) - self.T + 1
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.T]
        return torch.tensor(x, dtype=torch.float32)

## 2. VRNN Model Architecture

In [ ]:
class VRNN(nn.Module):
    def __init__(self, x_dim, z_dim, h_dim, n_layers, bias=False):
        super(VRNN, self).__init__()

        self.x_dim = x_dim
        self.z_dim = z_dim
        self.h_dim = h_dim
        self.n_layers = n_layers
    
        # Feature extractors
        self.phi_x = nn.Sequential(
            nn.Linear(x_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.ReLU()
        )

        self.phi_z = nn.Sequential(
            nn.Linear(z_dim, h_dim),
            nn.ReLU()
        )

        # Encoder q(z_t | x_t, h_{t-1})
        self.enc = nn.Sequential(
            nn.Linear(2 * h_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.ReLU()
        )
        self.enc_mu = nn.Linear(h_dim, z_dim)
        self.enc_logvar = nn.Linear(h_dim, z_dim)

        # Prior p(z_t | h_{t-1})
        self.prior = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.ReLU()
        )
        self.prior_mu = nn.Linear(h_dim, z_dim)
        self.prior_logvar = nn.Linear(h_dim, z_dim)

        # Decoder p(x_t | z_t, h_{t-1})
        self.dec = nn.Sequential(
            nn.Linear(2 * h_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.ReLU()
        )
        self.dec_mu = nn.Linear(h_dim, x_dim)
        self.dec_logvar = nn.Linear(h_dim, x_dim)

        # Recurrent dynamics
        self.rnn = nn.GRU(2 * h_dim, h_dim, n_layers, bias=bias)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def kl_gaussian(self, mu_q, logvar_q, mu_p, logvar_p):
        return 0.5 * torch.sum(
            logvar_p - logvar_q
            + (logvar_q.exp() + (mu_q - mu_p).pow(2)) / logvar_p.exp()
            - 1,
            dim=1
        )

    def nll_gaussian(self, x, mu, logvar):
        return 0.5 * torch.sum(
            logvar + (x - mu).pow(2) / logvar.exp(),
            dim=1
        )

    def forward(self, x, beta=1.0):
        T, B, _ = x.size()
        device = x.device
        h = torch.zeros(self.n_layers, B, self.h_dim, device=device)
        kld_loss = 0.0
        recon_loss = 0.0

        for t in range(T):
            x_t = x[t]
            phi_x_t = self.phi_x(x_t)
            h_last = h[-1]
            
            enc_input = torch.cat([phi_x_t, h_last], dim=1)
            enc_hidden = self.enc(enc_input)
            mu_q = self.enc_mu(enc_hidden)
            logvar_q = torch.clamp(self.enc_logvar(enc_hidden), -6, 6)

            prior_hidden = self.prior(h_last)
            mu_p = self.prior_mu(prior_hidden)
            logvar_p = torch.clamp(self.prior_logvar(prior_hidden), -6, 6)

            z_t = self.reparameterize(mu_q, logvar_q)
            phi_z_t = self.phi_z(z_t)

            dec_input = torch.cat([phi_z_t, h_last], dim=1)
            dec_hidden = self.dec(dec_input)
            dec_mu = self.dec_mu(dec_hidden)
            dec_logvar = torch.clamp(self.dec_logvar(dec_hidden), -6, 6)

            rnn_input = torch.cat([phi_x_t, phi_z_t], dim=1)
            _, h = self.rnn(rnn_input.unsqueeze(0), h)

            kld_loss += self.kl_gaussian(mu_q, logvar_q, mu_p, logvar_p).mean()
            recon_loss += self.nll_gaussian(x_t, dec_mu, dec_logvar).mean()

        total_loss = (beta * kld_loss + recon_loss) / T
        return total_loss, recon_loss / T, kld_loss / T

## 3. Training Logic

In [ ]:
def kl_annealing(epoch, warmup=20):
    return min(1.0, epoch / warmup)

def train(model, dataloader, optimizer, epochs=100, device='cpu'):
    model.train()
    os.makedirs("checkpoints", exist_ok=True)
    
    history = {"total_loss": [], "recon_loss": [], "kld_loss": []}
    
    pbar = tqdm(range(epochs), desc="Training")
    for epoch in pbar:
        epoch_total_loss, epoch_recon_loss, epoch_kld_loss = 0.0, 0.0, 0.0
        beta = kl_annealing(epoch)
        
        for x in dataloader:
            x = x.permute(1, 0, 2).to(device)
            optimizer.zero_grad()
            loss, recon_loss, kld_loss = model(x, beta)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            epoch_total_loss += loss.item()
            epoch_recon_loss += recon_loss.item()
            epoch_kld_loss += kld_loss.item()
        
        avg_loss = epoch_total_loss / len(dataloader)
        avg_recon = epoch_recon_loss / len(dataloader)
        avg_kld = epoch_kld_loss / len(dataloader)
        
        history["total_loss"].append(avg_loss)
        history["recon_loss"].append(avg_recon)
        history["kld_loss"].append(avg_kld)
        
        pbar.set_postfix({"Loss": f"{avg_loss:.4f}", "Beta": f"{beta:.4f}"})
        
        if (epoch + 1) % (epochs // 4) == 0:
            torch.save({'epoch': epoch+1, 'model_state_dict': model.state_dict(), 'history': history}, 
                       f"checkpoints/vrnn_epoch_{epoch+1}.pth")
    
    torch.save(model.state_dict(), "checkpoints/vrnn_final.pth")
    model.eval()
    return model, history

## 4. Main Execution
Adjust hyperparameters and file paths as needed.

In [ ]:
# Config
config = {
    "x_dim": 1,
    "z_dim": 16,
    "h_dim": 64,
    "n_layers": 2,
    "T": 40,
    "batch_size": 32,
    "lr": 1e-3,
    "epochs": 50, # Reduced for demo
    "file_path": "data/FPT Corp Stock Price History.csv", # UPDATE THIS PATH ON KAGGLE/COLAB
    "feature_columns": ["Change %"]
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure data folder exists for demo output (skip if file doesn't exist yet)
if os.path.exists(config['file_path']):
    dataset = FinanceDataset(T=config['T'], file_path=config['file_path'], feature_columns=config['feature_columns'])
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)

    model = VRNN(x_dim=config['x_dim'], z_dim=config['z_dim'], h_dim=config['h_dim'], n_layers=config['n_layers']).to(device)
    optimizer = Adam(model.parameters(), lr=config['lr'])

    model, history = train(model, dataloader, optimizer, epochs=config['epochs'], device=device)
    
    # Plot results
    plt.figure(figsize=(10, 5))
    plt.plot(history['total_loss'], label='Total Loss')
    plt.plot(history['recon_loss'], label='Recon Loss')
    plt.plot(history['kld_loss'], label='KLD Loss')
    plt.title("Training Loss History")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
else:
    print(f"Data file not found at {config['file_path']}. Please upload your CSV to the correct path.")